## **Uttrakhand Review Dataset Sentiment Analysis**

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
import os
import pandas as pd

# 1. Set the base path for Kaggle inputs
base_dir = '/kaggle/input/'
csv_files = []

# 2. The Detective: Walk through all folders to find .csv files
print("🕵️ Searching for CSV files...")
for root, dirs, files in os.walk(base_dir):
    for file in files:
        if file.endswith('.csv'):
            # Create the full path to the file
            full_path = os.path.join(root, file)
            csv_files.append(full_path)

print(f"✅ Found {len(csv_files)} CSV files!")

# 3. If we found files, let's merge them!
if len(csv_files) > 0:
    print(f"📂 First file found at: {csv_files[0]}")
    
    # Read all files into a list of DataFrames
    df_list = [pd.read_csv(f) for f in csv_files]
    
    # Stitch them together
    master_df = pd.concat(df_list, ignore_index=True)
    
    print(f"\n📊 Master DataFrame Shape: {master_df.shape}")
    print(f"\n📋 Columns available: {master_df.columns.tolist()}")
    
    # 4. Keep ONLY the columns we need (review and rating)
    # Note: We will check the exact spelling of 'rating' and 'review' from your output
    cols_to_keep = ['review', 'rating'] 
    
    # Check if columns exist before dropping others
    if all(col in master_df.columns for col in cols_to_keep):
        clean_df = master_df[cols_to_keep]
        print(f"\n✨ Cleaned DataFrame Shape: {clean_df.shape}")
        display(clean_df.head())
    else:
        print("⚠️ Could not find 'review' or 'rating' columns. Check the column names above.")
else:
    print("❌ No files found. Please check the dataset input.")

🕵️ Searching for CSV files...
✅ Found 1628 CSV files!
📂 First file found at: /kaggle/input/datasets/orionpax00/uttarakhand-hotel-reviews-india/Chamoli-20200601T161834Z-001/Chamoli/AR_Resort-Gauchar_Chamoli_District_Uttarakhand.csv

📊 Master DataFrame Shape: (96955, 6)

📋 Columns available: ['location', 'review', 'name', 'user_profile', 'userloc', 'rating']

✨ Cleaned DataFrame Shape: (96955, 2)


/tmp/ipykernel_49/4294900503.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  master_df = pd.concat(df_list, ignore_index=True)


,review,rating
0,good deal to this hotel and nice services.. ....,50.0
1,Hotel AR resort is located at main road. Best...,40.0
2,The only nice place to stay during my whole tr...,50.0
3,this is a small hotel located a little behind ...,40.0
4,This hotel is few KM's before rudraprayag on y...,30.0


In [7]:
master_df['rating'].unique()

array([50., 40., 30., 10., 20., nan])

In [8]:
master_df['rating'].value_counts()

rating
50.0    62931
40.0    22989
30.0     6320
10.0     2683
20.0     2028
Name: count, dtype: int64

In [10]:
master_df['rating'].isna().sum()

np.int64(4)

In [13]:
from sklearn.utils import resample

# 1. FIX THE WARNING: Create a true independent copy of the dataframe
# (In a real scenario, you would add .copy() to your previous step, 
# but we will do it here to be safe)
final_df = clean_df.copy()

In [14]:
# 2. Separate the dataset by sentiment
df_positive = final_df[final_df['sentiment'] == 'Positive']
df_neutral = final_df[final_df['sentiment'] == 'Neutral']
df_negative = final_df[final_df['sentiment'] == 'Negative']

# 3. Find the smallest count (Negative)
min_count = len(df_negative)
print(f"🎯 Target count for balancing: {min_count} reviews per class.")

🎯 Target count for balancing: 2683 reviews per class.


In [15]:

# 4. Randomly downsample the larger classes to match the smallest one
# random_state=42 ensures we get the same random selection every time we run it
df_positive_balanced = resample(df_positive, 
                                replace=False, # Do not replace (no duplicates)
                                n_samples=min_count, 
                                random_state=42)

df_neutral_balanced = resample(df_neutral, 
                               replace=False, 
                               n_samples=min_count, 
                               random_state=42)

In [16]:
# 5. Combine the balanced classes back together
balanced_df = pd.concat([df_positive_balanced, df_neutral_balanced, df_negative])

# 6. Shuffle the data so the model doesn't learn the order (all Positives, then Neutrals...)
balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [17]:
# 7. Verify the new distribution
print("\n✅ Final Balanced Dataset Distribution:")
print(balanced_df['sentiment'].value_counts())
print(f"\n📊 Total Reviews for Training: {len(balanced_df)}")


✅ Final Balanced Dataset Distribution:
sentiment
Positive    2683
Negative    2683
Neutral     2683
Name: count, dtype: int64

📊 Total Reviews for Training: 8049


In [18]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download necessary NLTK data (only needed once per session)
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [19]:
# 1. Define the cleaning function
def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    
    # Remove punctuation and special characters using Regex
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Tokenize (split sentence into words)
    tokens = word_tokenize(text)
    
    # Remove English stop words (e.g., 'the', 'is', 'in')
    stop_words = set(stopwords.words('english'))
    cleaned_tokens = [word for word in tokens if word not in stop_words]
    
    # Join tokens back into a single string
    return ' '.join(cleaned_tokens)

In [20]:
# 2. Apply the cleaning function to the 'review' column
# We create a new column 'cleaned_review' to keep the original safe
balanced_df['cleaned_review'] = balanced_df['review'].apply(clean_text)

# 3. Check the results
print("🧹 Text Cleaning Complete!\n")
print("Original Review:")
print(balanced_df['review'].iloc[0])
print("\nCleaned Review:")
print(balanced_df['cleaned_review'].iloc[0])

🧹 Text Cleaning Complete!

Original Review:
At Hotel Classic Ladakh you will find more than a hotel manager: you will find a friend. Mr. Ashok goes to great lengths to make your stay comfortable and pleasant, helped by a great team. Rooms are spacious, with a wonderful view on the mountains. The food served at the restaurant is also very tasty: we had some of the best momos in town!  We wish we could have stayed longer, and cannot thank Mr. Ashok enough for all the help he has given us. Wishing him all the best, Giuseppe

Cleaned Review:
hotel classic ladakh find hotel manager find friend mr ashok goes great lengths make stay comfortable pleasant helped great team rooms spacious wonderful view mountains food served restaurant also tasty best momos town wish could stayed longer thank mr ashok enough help given us wishing best giuseppe


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Define our Features (X) and Target (y)
X = balanced_df['cleaned_review']
y = balanced_df['sentiment']

In [26]:
# 2. Initialize the TF-IDF Vectorizer
# max_features=5000 keeps only the top 5000 most important words to save memory
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')

In [27]:
# 3. Transform the text into a numerical matrix (X_tfidf)
X_tfidf = tfidf.fit_transform(X)

In [29]:

# 4. Split the data: 80% for Training, 20% for Testing
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)

In [30]:
# 5. Verify the shapes
print("📊 Data Split Complete!")
print(f"Training Data Shape: {X_train.shape} (Rows, Features)")
print(f"Testing Data Shape:  {X_test.shape} (Rows, Features)")

📊 Data Split Complete!
Training Data Shape: (6439, 5000) (Rows, Features)
Testing Data Shape:  (1610, 5000) (Rows, Features)


# **SVM**

In [31]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

# 1. Initialize the Linear SVM model
# We set a random_state so our results are reproducible
svm_model = LinearSVC(random_state=42, max_iter=10000)

# 2. Train the model on the training data
print("🏋️ Training the Linear SVM Model...")
svm_model.fit(X_train, y_train)

# 3. Make predictions on the hidden test data
print("🔮 Making predictions on test data...")
y_pred = svm_model.predict(X_test)

# 4. Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"\n🎯 Model Accuracy: {accuracy * 100:.2f}%\n")

print("📊 Classification Report:")
print(classification_report(y_test, y_pred))

🏋️ Training the Linear SVM Model...
🔮 Making predictions on test data...

🎯 Model Accuracy: 77.76%

📊 Classification Report:
              precision    recall  f1-score   support

    Negative       0.80      0.82      0.81       536
     Neutral       0.70      0.68      0.69       537
    Positive       0.83      0.84      0.83       537

    accuracy                           0.78      1610
   macro avg       0.78      0.78      0.78      1610
weighted avg       0.78      0.78      0.78      1610



## **Naive Bayes**

In [32]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score

# 1. Initialize the Multinomial Naive Bayes model
nb_model = MultinomialNB()

# 2. Train the model on our existing training data
print("🏋️ Training Multinomial Naive Bayes...")
nb_model.fit(X_train, y_train)

# 3. Make predictions on the test data
print("🔮 Making predictions...")
y_pred_nb = nb_model.predict(X_test)

# 4. Evaluate the model
accuracy_nb = accuracy_score(y_test, y_pred_nb)
print(f"\n🎯 Naive Bayes Accuracy: {accuracy_nb * 100:.2f}%\n")

print("📊 Classification Report:")
print(classification_report(y_test, y_pred_nb))

🏋️ Training Multinomial Naive Bayes...
🔮 Making predictions...

🎯 Naive Bayes Accuracy: 79.50%

📊 Classification Report:
              precision    recall  f1-score   support

    Negative       0.83      0.83      0.83       536
     Neutral       0.70      0.72      0.71       537
    Positive       0.86      0.84      0.85       537

    accuracy                           0.80      1610
   macro avg       0.80      0.80      0.80      1610
weighted avg       0.80      0.80      0.80      1610



## **Logistic Regression**

In [33]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# 1. Initialize the Logistic Regression model
# max_iter=10000 ensures the model has enough time to find the best mathematical boundary
lr_model = LogisticRegression(max_iter=10000, random_state=42)

# 2. Train the model on our existing training data
print("🏋️ Training Logistic Regression...")
lr_model.fit(X_train, y_train)

# 3. Make predictions on the test data
print("🔮 Making predictions...")
y_pred_lr = lr_model.predict(X_test)

# 4. Evaluate the model
accuracy_lr = accuracy_score(y_test, y_pred_lr)
print(f"\n🎯 Logistic Regression Accuracy: {accuracy_lr * 100:.2f}%\n")

print("📊 Classification Report:")
print(classification_report(y_test, y_pred_lr))

🏋️ Training Logistic Regression...
🔮 Making predictions...

🎯 Logistic Regression Accuracy: 80.00%

📊 Classification Report:
              precision    recall  f1-score   support

    Negative       0.82      0.84      0.83       536
     Neutral       0.72      0.72      0.72       537
    Positive       0.86      0.84      0.85       537

    accuracy                           0.80      1610
   macro avg       0.80      0.80      0.80      1610
weighted avg       0.80      0.80      0.80      1610



In [34]:
import joblib
import os

# 1. Save the TF-IDF Vectorizer
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
print("✅ Saved: tfidf_vectorizer.pkl")

# 2. Save the Winning Logistic Regression Model
joblib.dump(lr_model, 'sentiment_lr_model.pkl')
print("✅ Saved: sentiment_lr_model.pkl")

# 3. Verify the files were created
print("\n📂 Files ready for your GitHub repo:")
print([f for f in os.listdir('.') if f.endswith('.pkl')])

✅ Saved: tfidf_vectorizer.pkl
✅ Saved: sentiment_lr_model.pkl

📂 Files ready for your GitHub repo:
['sentiment_lr_model.pkl', 'tfidf_vectorizer.pkl']
